# MIST · Attention Gates + Boundary Loss — RunPod Training
**Branch**: `ag-bl-good-baseline` | **Model**: `MIST_CAM` with Attention Gates + Boundary-Weighted Loss

**Changes vs. baseline**:
- `Block_decoder` and `Block_decoder1`: gated skip connections via `AttentionGate` — uses decoder state to score each encoder skip position before concatenation, suppressing irrelevant regions
- Loss: `0.7 × Dice + 0.3 × CE + 0.2 × BoundaryLoss` per powerset subset; BoundaryLoss is boundary-weighted CE with boundary pixels up-weighted by `w=3`
- Validation: mean per-class Dice (RV / Myo / LV) — not binary foreground Dice
- No medpy: all metrics use scipy drop-in replacements

**Architecture unchanged from baseline**: `BottleneckBlock`, `Bottleneck_decoder`, `Dilated_Conv` (SWC d=2,3 concat), SSAM (`CBAM`) in `Transformer`, 3 active decoder outputs (block-1 excluded from supervision, paper §3.3).

**Training matches paper Section 3.7**: AdamW lr=1e-4, wd=1e-4, batch=12, img=256×256, 300 epochs, fixed LR, powerset mutation on 3 outputs (7 non-empty subsets).

**Run cells in order. No kernel restart needed.**

In [ ]:
# Cell 2 — GPU check
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'No GPU detected!'
print('GPU check passed')

In [ ]:
# Cell 3 — Install dependencies  (no medpy — replaced with scipy drop-ins)
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args),
                       capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-600:]

pkgs = [
    'scipy>=1.14.0',
    'timm==0.9.12',
    'SimpleITK',
    'seaborn',
    'segmentation-mask-overlay',
    'thop',
    'scikit-image',
    'einops',
    'tensorboardX',
    'tqdm',
    'gdown',
    'matplotlib',
    'pandas',
]
for spec in pkgs:
    code, out = pip('install', spec, '--quiet')
    status = 'OK    ' if code == 0 else 'FAILED'
    print(f'  {status}  {spec}')
    if code != 0:
        print(out)

print('\nAll dependencies installed.')

In [ ]:
# Cell 4 — Smoke test imports
import numpy as np
import scipy
import torch
import timm

print(f'numpy     {np.__version__}')
print(f'scipy     {scipy.__version__}')
print(f'torch     {torch.__version__}  (CUDA {torch.version.cuda})')
print(f'timm      {timm.__version__}')
print('All imports OK')

In [ ]:
# Cell 5 — Clone / update repo  (branch: ag-bl-good-baseline)
import subprocess, os, sys

REPO_URL = 'https://github.com/biancafabian/MIST.git'
BRANCH   = 'ag-bl-good-baseline'
REPO_DIR = '/workspace/MIST'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'Repo exists at {REPO_DIR}, syncing {BRANCH}...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH],
                   capture_output=True)
    r = subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard',
                        f'origin/{BRANCH}'], capture_output=True, text=True)
    print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
else:
    print(f'Cloning {BRANCH}...')
    r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout.strip(), r.stderr.strip())
    assert r.returncode == 0, f'git clone failed: {r.stderr}'

# Verify key files exist
for fname in ['lib/MIST.py', 'lib/networks.py', 'ACDC_train_test.py',
              'utils/utils.py', 'utils/dataset_ACDC.py']:
    path = os.path.join(REPO_DIR, fname)
    ok = 'OK     ' if os.path.exists(path) else 'MISSING'
    print(f'  {ok}  {fname}')

# Confirm branch-specific additions are present
with open(os.path.join(REPO_DIR, 'lib', 'MIST.py')) as fh:
    mist_src = fh.read()
assert 'class AttentionGate' in mist_src, \
    'AttentionGate not found in lib/MIST.py — push changes first'
assert 'class BottleneckBlock' in mist_src, \
    'BottleneckBlock missing — base architecture not on this branch'
assert 'self.ssam' in mist_src, \
    'SSAM not active in Transformer — check lib/MIST.py'
print('AttentionGate       confirmed in lib/MIST.py')
print('BottleneckBlock     confirmed in lib/MIST.py')
print('SSAM (self.ssam)    confirmed in lib/MIST.py')

with open(os.path.join(REPO_DIR, 'utils', 'utils.py')) as fh:
    utils_src = fh.read()
assert 'class BoundaryLoss' in utils_src, \
    'BoundaryLoss not found in utils/utils.py — push changes first'
assert 'from medpy' not in utils_src, \
    'medpy import still present in utils/utils.py — remove it'
print('BoundaryLoss        confirmed in utils/utils.py')
print('medpy-free          confirmed')

with open(os.path.join(REPO_DIR, 'lib', 'networks.py')) as fh:
    nets_src = fh.read()
assert 'class MIST_CAM' in nets_src, 'MIST_CAM not found in lib/networks.py'
assert 'out_head1' not in nets_src, \
    'out_head1 present — should have 3 outputs (block-1 excluded), not 4'
print('MIST_CAM (3 outputs, block-1 excluded) confirmed in lib/networks.py')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

In [ ]:
# Cell 7 — Download ACDC dataset (skip if already present)
import os, subprocess, sys, zipfile

DATA_ROOT = './data/ACDC'

if (os.path.isdir(DATA_ROOT) and
        os.path.isdir(os.path.join(DATA_ROOT, 'train')) and
        len(os.listdir(os.path.join(DATA_ROOT, 'train'))) > 0):
    print(f'ACDC data found at {DATA_ROOT}, skipping download.')
else:
    ACDC_FILE_ID = 'YOUR_GDRIVE_FILE_ID_HERE'   # <-- REPLACE THIS
    assert ACDC_FILE_ID != 'YOUR_GDRIVE_FILE_ID_HERE', \
        'Set ACDC_FILE_ID to the Google Drive file ID before running!'

    import gdown
    print('Downloading ACDC dataset from Google Drive...')
    gdown.download(f'https://drive.google.com/uc?id={ACDC_FILE_ID}',
                   'ACDC_dataset.zip', quiet=False)
    print('Extracting...')
    with zipfile.ZipFile('ACDC_dataset.zip', 'r') as z:
        z.extractall('.')
    os.remove('ACDC_dataset.zip')
    print('Done.')

print('\nData layout check:')
for sub in ['train', 'valid', 'test', 'lists_ACDC']:
    p = os.path.join(DATA_ROOT, sub)
    if os.path.isdir(p):
        n = len(os.listdir(p))
        print(f'  {sub:15s}  {n} entries')
    else:
        print(f'  {sub:15s}  MISSING')

## Data Preprocessing
Run **Cell 7c** only if you have raw ACDC `.nii.gz` files (original challenge download). Skip if `./data/ACDC/` is already populated.

In [ ]:
# Cell 7c — Preprocess raw ACDC .nii.gz files  (skip if already preprocessed)
# Set RAW_DIR to the folder containing patient001/ … patient100/

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'nibabel', '--quiet'], check=True)

import os
import numpy as np
import nibabel as nib
from tqdm import tqdm

RAW_DIR = '/workspace/ACDC_raw/training'   # <-- set this
OUT_DIR = './data/ACDC'

assert os.path.isdir(RAW_DIR), f'RAW_DIR not found: {RAW_DIR}'

TRAIN_IDS = list(range(1,  71))
VALID_IDS = list(range(71, 81))
TEST_IDS  = list(range(81, 101))

for sub in ['train', 'valid', 'test', 'lists_ACDC']:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)


def read_ed_es(patient_dir):
    cfg = os.path.join(patient_dir, 'Info.cfg')
    ed = es = None
    with open(cfg) as fh:
        for line in fh:
            if line.startswith('ED:'): ed = int(line.split(':')[1].strip())
            elif line.startswith('ES:'): es = int(line.split(':')[1].strip())
    return ed, es


def normalize_volume(vol):
    vmin, vmax = float(vol.min()), float(vol.max())
    if vmax - vmin < 1e-8:
        return np.zeros_like(vol, dtype=np.float32)
    return ((vol - vmin) / (vmax - vmin)).astype(np.float32)


def process_patient(pid, split, train_names, valid_names, test_names):
    pname = f'patient{pid:03d}'
    pdir  = os.path.join(RAW_DIR, pname)
    if not os.path.isdir(pdir):
        print(f'  WARNING: {pdir} not found — skipping')
        return
    ed_frame, es_frame = read_ed_es(pdir)
    for frame_num in [ed_frame, es_frame]:
        fstr     = f'{frame_num:02d}'
        img_path = os.path.join(pdir, f'{pname}_frame{fstr}.nii.gz')
        lbl_path = os.path.join(pdir, f'{pname}_frame{fstr}_gt.nii.gz')
        if not os.path.exists(img_path) or not os.path.exists(lbl_path):
            continue
        img_vol = np.transpose(nib.load(img_path).get_fdata().astype(np.float32), (2, 0, 1))
        lbl_vol = np.transpose(nib.load(lbl_path).get_fdata().astype(np.uint8),   (2, 0, 1))
        img_vol = normalize_volume(img_vol)
        if split == 'test':
            fname = f'{pname}_frame{fstr}.npz'
            np.savez_compressed(os.path.join(OUT_DIR, 'test', fname),
                                img=img_vol, label=lbl_vol)
            test_names.append(fname)
        else:
            subdir = os.path.join(OUT_DIR, split)
            for si in range(img_vol.shape[0]):
                fname = f'{pname}_frame{fstr}_slice{si:03d}.npz'
                np.savez_compressed(os.path.join(subdir, fname),
                                    img=img_vol[si], label=lbl_vol[si])
                (train_names if split == 'train' else valid_names).append(fname)


train_names, valid_names, test_names = [], [], []

print('Train — patients 001-070')
for pid in tqdm(TRAIN_IDS): process_patient(pid, 'train', train_names, valid_names, test_names)
print(f'  {len(train_names)} slices')

print('Valid — patients 071-080')
for pid in tqdm(VALID_IDS): process_patient(pid, 'valid', train_names, valid_names, test_names)
print(f'  {len(valid_names)} slices')

print('Test  — patients 081-100')
for pid in tqdm(TEST_IDS): process_patient(pid, 'test', train_names, valid_names, test_names)
print(f'  {len(test_names)} volumes')

lists_dir = os.path.join(OUT_DIR, 'lists_ACDC')
with open(os.path.join(lists_dir, 'train.txt'), 'w') as f: f.write('\n'.join(train_names) + '\n')
with open(os.path.join(lists_dir, 'valid.txt'), 'w') as f: f.write('\n'.join(valid_names) + '\n')
with open(os.path.join(lists_dir, 'test.txt'),  'w') as f: f.write('\n'.join(test_names)  + '\n')
print('Preprocessing complete.')

In [ ]:
# Cell 8 — Verify dataset
import os, glob
import numpy as np

TRAIN_DIR = './data/ACDC/train'
VALID_DIR = './data/ACDC/valid'
TEST_DIR  = './data/ACDC/test'

train_files = glob.glob(os.path.join(TRAIN_DIR, '*.npz'))
valid_files = glob.glob(os.path.join(VALID_DIR, '*.npz'))
test_files  = glob.glob(os.path.join(TEST_DIR,  '*.npz'))
print(f'Train slices : {len(train_files):4d}  (expected ~1304)')
print(f'Valid slices : {len(valid_files):4d}  (expected ~182)')
print(f'Test  items  : {len(test_files):4d}')

if train_files:
    d = np.load(train_files[0])
    img, lbl = d['img'], d['label']
    print(f'Train sample  img={img.shape}  label={lbl.shape}  '
          f'range=[{img.min():.3f}, {img.max():.3f}]  '
          f'classes={sorted(set(lbl.ravel().tolist()))}')

if test_files:
    d = np.load(test_files[0])
    img, lbl = d['img'], d['label']
    print(f'Test  sample  img={img.shape}  label={lbl.shape}  ndim={img.ndim}')

print('Dataset OK')

In [ ]:
# Cell 9 — Pre-download MaxViT pretrained weights
import os, torch

WEIGHTS_PATH = './pretrained_pth/maxvit/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth'
WEIGHTS_URL  = ('https://github.com/rwightman/pytorch-image-models/releases/'
                'download/v0.1-weights-maxx/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth')

if os.path.exists(WEIGHTS_PATH):
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Weights already present ({size_mb:.0f} MB): {WEIGHTS_PATH}')
else:
    os.makedirs(os.path.dirname(WEIGHTS_PATH), exist_ok=True)
    print('Downloading MaxViT weights...')
    torch.hub.download_url_to_file(WEIGHTS_URL, WEIGHTS_PATH)
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Saved ({size_mb:.0f} MB): {WEIGHTS_PATH}')
print('MaxViT weights OK')

In [ ]:
# Cell 11 — Training configuration
# Paper Section 3.7: AdamW lr=1e-4, wd=1e-4, batch=12, img=256, 300 epochs, fixed LR.
BATCH_SIZE   = 12
LR           = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 300
IMG_SIZE     = 256
NUM_CLASSES  = 4       # BG, RV, Myo, LV
SEED         = 2222
BOUNDARY_W   = 3       # boundary pixel up-weight in BoundaryLoss

LC1, LC2, LC3 = 0.7, 0.3, 0.2  # loss weights: LC1*Dice + LC2*CE + LC3*BoundaryLoss

DATA_DIR = './data/ACDC'
LIST_DIR = './data/ACDC/lists_ACDC'
TEST_DIR = './data/ACDC/test'
SAVE_DIR = './model_pth'

print(f'batch={BATCH_SIZE}  lr={LR}  wd={WEIGHT_DECAY}  epochs={MAX_EPOCHS}')
print(f'img={IMG_SIZE}x{IMG_SIZE}  classes={NUM_CLASSES}  seed={SEED}')
print(f'loss = {LC1}*Dice + {LC2}*CE + {LC3}*BoundaryLoss(w={BOUNDARY_W})  (powerset mutation, 3 outputs, 7 subsets)')

In [ ]:
# Cell 12 — Initialise model, data loaders, losses, optimiser
import os, sys, time, random
import numpy as np
import torch
import torch.optim as optim
from torch.nn.modules.loss import CrossEntropyLoss
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast
from scipy.ndimage import zoom

REPO_DIR = '/workspace/MIST'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)

from lib.networks import MIST_CAM
from utils.utils import DiceLoss, BoundaryLoss, powerset, _dc
from utils.dataset_ACDC import ACDCdataset, RandomGenerator

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Snapshot directory
run_id        = time.strftime('%H%M%S')
snapshot_path = os.path.join(SAVE_DIR, f'MIST_CAM_AG_BndLoss_{IMG_SIZE}_run{run_id}')
os.makedirs(snapshot_path, exist_ok=True)
print(f'Snapshot dir: {snapshot_path}')

# Model (MaxViT encoder + CAM decoder with AttentionGates + SSAM)
net = MIST_CAM(
    n_class=NUM_CLASSES,
    img_size_s1=(IMG_SIZE, IMG_SIZE),
    img_size_s2=(224, 224),
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
).cuda()
n_params = sum(p.numel() for p in net.parameters()) / 1e6
print(f'Model: MIST_CAM (AttentionGates + SSAM)  params={n_params:.1f}M')

# torch.compile: fuses ops for ~15-20% faster iterations (PyTorch 2.0+).
# First epoch will be slower (JIT compilation), subsequent epochs benefit.
if hasattr(torch, 'compile'):
    try:
        net = torch.compile(net, mode='reduce-overhead')
        print('torch.compile enabled')
    except Exception as e:
        print(f'torch.compile skipped: {e}')
else:
    print('torch.compile not available (PyTorch < 2.0), continuing without it')

# Losses
ce_loss       = CrossEntropyLoss()
dice_loss     = DiceLoss(NUM_CLASSES)
boundary_loss = BoundaryLoss(NUM_CLASSES, w=BOUNDARY_W)

# Data loaders
train_dataset = ACDCdataset(
    DATA_DIR, LIST_DIR, split='train',
    transform=transforms.Compose([RandomGenerator(output_size=[IMG_SIZE, IMG_SIZE])]))
val_dataset = ACDCdataset(DATA_DIR, LIST_DIR, split='valid')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
valloader    = DataLoader(val_dataset,   batch_size=1,          shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)
print(f'Train: {len(train_dataset)} slices  |  Val: {len(val_dataset)} slices')
print(f'Train iters/epoch: {len(train_loader)}')

# Optimiser — paper Section 3.7: AdamW, fixed LR
optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = GradScaler()

# Save run config
with open(os.path.join(snapshot_path, 'config.txt'), 'w') as f:
    f.write('model=MIST_CAM\n')
    f.write('decoder=CAM_with_AttentionGates\n')
    f.write('ssam=enabled\n')
    f.write(f'boundary_w={BOUNDARY_W}\n')
    f.write(f'batch={BATCH_SIZE}\n')
    f.write(f'lr={LR}\n')
    f.write(f'weight_decay={WEIGHT_DECAY}\n')
    f.write('lr_schedule=fixed\n')
    f.write(f'epochs={MAX_EPOCHS}\n')
    f.write(f'img_size={IMG_SIZE}\n')
    f.write(f'seed={SEED}\n')
    f.write(f'classes={NUM_CLASSES}\n')
    f.write(f'loss={LC1}*Dice+{LC2}*CE+{LC3}*BoundaryLoss(w={BOUNDARY_W})_powerset_mutation\n')
print('config.txt saved')

In [ ]:
# Cell 13 — Training loop
# Loss per 7 non-empty powerset subsets of 3 outputs: LC1*Dice + LC2*CE + LC3*BoundaryLoss
# Validation: mean per-class Dice (RV/Myo/LV), not binary foreground Dice.
# Checkpoints: last.pth (every epoch, resumable) + best.pth (best mean val Dice).
from tqdm import tqdm

CLASS_NAMES = ['RV', 'Myo', 'LV']

l  = [0, 1, 2]   # 3 active outputs (block-1 excluded per paper §3.3)
ss = [s for s in powerset(l)]
print(f'Active outputs: {len(l)}  |  Powerset subsets: {len(ss)}')


def dice_cls(pred, gt, c):
    """Per-class Dice for class c."""
    p = (pred == c).astype(bool)
    g = (gt   == c).astype(bool)
    inter = np.count_nonzero(p & g)
    denom = np.count_nonzero(p) + np.count_nonzero(g)
    return 2.0 * inter / denom if denom else 0.0


def do_val():
    """Returns (mean_dice, [rv_dice, myo_dice, lv_dice])."""
    net.eval()
    cls_sums = [0.0] * (NUM_CLASSES - 1)
    with torch.no_grad():
        for vb in valloader:
            img = vb['image'].squeeze(0).cpu().numpy()
            lbl = vb['label'].squeeze(0).cpu().numpy()
            h, w = img.shape[0], img.shape[1]
            if h != IMG_SIZE or w != IMG_SIZE:
                img = zoom(img, (IMG_SIZE / h, IMG_SIZE / w), order=3)
            t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float().cuda()
            with autocast():
                P = net(t)
            out = sum(P)
            out = torch.softmax(out, dim=1).argmax(dim=1).squeeze(0).cpu().numpy()
            if h != IMG_SIZE or w != IMG_SIZE:
                out = zoom(out, (h / IMG_SIZE, w / IMG_SIZE), order=0)
            for ci in range(NUM_CLASSES - 1):
                cls_sums[ci] += dice_cls(out, lbl, ci + 1)
    net.train()
    n = len(valloader)
    per_cls = [s / n for s in cls_sums]
    return float(np.mean(per_cls)), per_cls


# History
Loss, Loss_d, Loss_c, Loss_b = [], [], [], []
ValDice, ValDice_cls = [], [[] for _ in range(NUM_CLASSES - 1)]

best_dcs   = 0.80
best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
iter_num   = 0

print(f'Training {MAX_EPOCHS} epochs | {len(train_loader)} iters/epoch')
print(f'Snapshot: {snapshot_path}')

for epoch in tqdm(range(MAX_EPOCHS)):
    net.train()
    ep_loss = ep_ld = ep_lc = ep_lb = 0.0

    for sampled_batch in train_loader:
        imgs   = sampled_batch['image'].float().cuda()
        labels = sampled_batch['label'].float().cuda()

        with autocast():
            P    = net(imgs)
            loss = 0.0
            for s in ss:
                if s == []:
                    continue
                iout  = sum(P[idx] for idx in s)
                ld    = dice_loss(iout, labels, softmax=True)
                lc    = ce_loss(iout, labels.long())
                lb    = boundary_loss(iout, labels.long())
                loss += LC1 * ld + LC2 * lc + LC3 * lb
                ep_ld += (LC1 * ld).item()
                ep_lc += (LC2 * lc).item()
                ep_lb += (LC3 * lb).item()

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        iter_num  += 1
        ep_loss   += loss.item()
        if iter_num % 100 == 0:
            tqdm.write(f'  iter {iter_num:5d}  loss {loss.item():.4f}  lr {LR:.2e}')

    n_iters = len(train_loader)
    Loss.append(ep_loss / n_iters)
    Loss_d.append(ep_ld  / n_iters)
    Loss_c.append(ep_lc  / n_iters)
    Loss_b.append(ep_lb  / n_iters)

    # Resumable checkpoint
    torch.save({
        'epoch':                epoch,
        'iter_num':             iter_num,
        'model_state_dict':     net.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_dcs':             best_dcs,
    }, os.path.join(snapshot_path, 'last.pth'))

    avg_dcs, per_cls = do_val()
    ValDice.append(avg_dcs)
    for ci, v in enumerate(per_cls):
        ValDice_cls[ci].append(v)

    tqdm.write(f'Epoch {epoch+1:3d}/{MAX_EPOCHS}  '
               f'loss={Loss[-1]:.4f}  '
               f'val_dc={avg_dcs:.4f} (RV={per_cls[0]:.3f} Myo={per_cls[1]:.3f} LV={per_cls[2]:.3f})  '
               f'best={best_dcs:.4f}')

    if avg_dcs > best_dcs:
        best_dcs   = avg_dcs
        best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
        torch.save(net.state_dict(), os.path.join(snapshot_path, 'best.pth'))
        tqdm.write(f'  -> New best: {best_dcs:.4f}')

torch.save(best_state, os.path.join(snapshot_path, 'best.pth'))
print(f'Training complete.  Best val Dice: {best_dcs:.4f}')
print(f'Checkpoint: {snapshot_path}/best.pth')

In [ ]:
# Cell 14 — Training loss + mean validation Dice
import matplotlib.pyplot as plt

best_epoch = int(np.argmax(ValDice))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(Loss, color='steelblue', linewidth=1.2)
axes[0].set_title('Train Loss per Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(ValDice, color='seagreen', linewidth=1.2, label='Mean val Dice')
axes[1].axhline(max(ValDice), color='crimson', linestyle='--', alpha=0.7,
                label=f'Best: {max(ValDice):.4f} @ epoch {best_epoch + 1}')
axes[1].set_title('Validation Dice (mean per-class)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Dice', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'training_curves.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')
print(f'Best val Dice: {max(ValDice):.4f}  at epoch {best_epoch + 1}')

In [ ]:
# Cell 14b — Per-class validation Dice curves (RV / Myo / LV)
import matplotlib.pyplot as plt

COLORS = {'RV': '#e63946', 'Myo': '#f4a261', 'LV': '#2a9d8f'}

fig, ax = plt.subplots(figsize=(10, 4))

for ci, cname in enumerate(CLASS_NAMES):
    vals       = ValDice_cls[ci]
    best_e     = int(np.argmax(vals))
    ax.plot(vals, color=COLORS[cname], linewidth=1.4, label=f'{cname} (best {max(vals):.3f} @ ep {best_e+1})')
    ax.axhline(max(vals), color=COLORS[cname], linestyle=':', alpha=0.4)

ax.plot(ValDice, color='black', linewidth=1.8, linestyle='--', alpha=0.6, label='Mean')

ax.set_title('Per-Class Validation Dice over Training', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Dice', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'val_dice_per_class.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# Cell 14c — Loss component breakdown (Dice / CE / Boundary contributions)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Stacked area: each epoch's loss split into three components
epochs = list(range(1, len(Loss) + 1))
axes[0].stackplot(
    epochs,
    Loss_d, Loss_c, Loss_b,
    labels=[f'Dice (×{LC1})', f'CE (×{LC2})', f'Boundary (×{LC3})'],
    colors=['#4895ef', '#f72585', '#7b2d8b'],
    alpha=0.75)
axes[0].set_title('Loss Component Breakdown (stacked)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss contribution', fontsize=11)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(True, alpha=0.2)

# Individual component curves
axes[1].plot(epochs, Loss_d, color='#4895ef', linewidth=1.4, label=f'Dice ×{LC1}')
axes[1].plot(epochs, Loss_c, color='#f72585', linewidth=1.4, label=f'CE ×{LC2}')
axes[1].plot(epochs, Loss_b, color='#7b2d8b', linewidth=1.4, label=f'Boundary ×{LC3}')
axes[1].set_title('Loss Components (individual)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss contribution', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'loss_components.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# Cell 14d — Qualitative sample predictions (input | ground truth | prediction)
# Picks 5 validation slices that contain all 3 foreground classes.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Class colours (RGBA, matching ACDC convention)
CMAP_COLORS = [
    [0,   0,   0,   0  ],   # 0: background — transparent
    [220, 50,  50,  200],   # 1: RV  — red
    [255, 165, 0,   200],   # 2: Myo — orange
    [50,  180, 100, 200],   # 3: LV  — green
]

def label_to_rgba(lbl):
    h, w  = lbl.shape
    rgba  = np.zeros((h, w, 4), dtype=np.uint8)
    for c, col in enumerate(CMAP_COLORS):
        mask = lbl == c
        rgba[mask] = col
    return rgba


net.eval()
samples, count = [], 0
with torch.no_grad():
    for vb in valloader:
        img = vb['image'].squeeze(0).cpu().numpy()
        lbl = vb['label'].squeeze(0).cpu().numpy().astype(int)
        if not all((lbl == c).any() for c in range(1, NUM_CLASSES)):
            continue  # skip slices missing any class
        h, w = img.shape[0], img.shape[1]
        inp  = zoom(img, (IMG_SIZE / h, IMG_SIZE / w), order=3) if (h != IMG_SIZE or w != IMG_SIZE) else img
        t    = torch.from_numpy(inp).unsqueeze(0).unsqueeze(0).float().cuda()
        with autocast():
            P = net(t)
        out  = sum(P)
        pred = torch.softmax(out, dim=1).argmax(dim=1).squeeze(0).cpu().numpy()
        if h != IMG_SIZE or w != IMG_SIZE:
            pred = zoom(pred, (h / IMG_SIZE, w / IMG_SIZE), order=0).astype(int)
        samples.append((img, lbl, pred))
        count += 1
        if count >= 5:
            break
net.train()

n_samples = len(samples)
fig, axes = plt.subplots(n_samples, 3, figsize=(12, 3.5 * n_samples))
if n_samples == 1:
    axes = axes[np.newaxis, :]

col_titles = ['Input Image', 'Ground Truth', 'Prediction']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold', pad=8)

for row, (img, lbl, pred) in enumerate(samples):
    axes[row, 0].imshow(img, cmap='gray', interpolation='bilinear')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(img, cmap='gray', interpolation='bilinear')
    axes[row, 1].imshow(label_to_rgba(lbl), interpolation='nearest')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(img, cmap='gray', interpolation='bilinear')
    axes[row, 2].imshow(label_to_rgba(pred), interpolation='nearest')
    axes[row, 2].axis('off')

legend_patches = [
    mpatches.Patch(color=tuple(c / 255 for c in CMAP_COLORS[1][:3]), label='RV'),
    mpatches.Patch(color=tuple(c / 255 for c in CMAP_COLORS[2][:3]), label='Myo'),
    mpatches.Patch(color=tuple(c / 255 for c in CMAP_COLORS[3][:3]), label='LV'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           fontsize=11, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

plt.suptitle('MIST-CAM + Attention Gates: Sample Predictions', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'sample_predictions.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# Cell 15 — Package checkpoint for download
import shutil, os
from IPython.display import FileLink, display

zip_base = f'/workspace/MIST_CAM_AG_BndLoss_{IMG_SIZE}_run{run_id}'
print(f'Zipping {snapshot_path} ...')
shutil.make_archive(zip_base, 'zip', snapshot_path)
zip_path = zip_base + '.zip'
size_mb  = os.path.getsize(zip_path) / 1e6
print(f'Created: {zip_path}  ({size_mb:.0f} MB)')
display(FileLink(zip_path))

In [ ]:
# Cell 15b — Optional: upload to Google Drive with rclone
# Configure rclone first: !rclone config
RUN_UPLOAD    = False
REMOTE        = 'gdrive'
GDRIVE_FOLDER = 'MIST_AG_BndLoss_results'

if RUN_UPLOAD:
    import subprocess
    r = subprocess.run(
        ['rclone', 'copy', zip_path, f'{REMOTE}:{GDRIVE_FOLDER}/', '-v'],
        capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print('rclone FAILED:', r.stderr)
else:
    print('Upload skipped (set RUN_UPLOAD=True to enable).')

## Volume-Level Inference
Run after training. Loads `best.pth`, runs per-slice inference, then aggregates by volume and reports **per-class Dice and HD95** — the metrics used in paper Table 1.

Paper target (baseline `MIST_CAM`): **Mean Dice 92.56%** — RV 91.23, Myo 90.31, LV 96.14.

In [ ]:
# Cell 16 — Volume-level inference (per-class Dice + HD95, no medpy)
import os, re, glob, sys, collections
import numpy as np
import torch
from tqdm import tqdm
from scipy.ndimage import zoom, binary_erosion, distance_transform_edt

REPO_DIR = '/workspace/MIST'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)


def _dc_vol(p, g):
    p, g  = p.astype(bool), g.astype(bool)
    inter = np.count_nonzero(p & g)
    denom = np.count_nonzero(p) + np.count_nonzero(g)
    return 2.0 * inter / float(denom) if denom else 0.0


def _hd95_vol(p, g):
    p, g = p.astype(bool), g.astype(bool)
    if not p.any() or not g.any():
        return 0.0
    rb = p ^ binary_erosion(p)
    sb = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~g)[rb]
    d2 = distance_transform_edt(~p)[sb]
    return float(np.percentile(np.hstack([d1, d2]), 95))


# Load model
CHECKPOINT = os.path.join(snapshot_path, 'best.pth')
print(f'Loading: {CHECKPOINT}')

from lib.networks import MIST_CAM as _MIST_CAM
net_inf = _MIST_CAM(
    n_class=NUM_CLASSES,
    img_size_s1=(IMG_SIZE, IMG_SIZE),
    img_size_s2=(224, 224),
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
).cuda()
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.replace('_orig_mod.', '', 1): v for k, v in state.items()}
    print('Stripped torch.compile prefix from state dict')
net_inf.load_state_dict(state)
net_inf.eval()
print('Model loaded')

# Group test slices by patient+frame
test_npz = sorted(glob.glob(os.path.join(TEST_DIR, '*.npz')))
print(f'Test files: {len(test_npz)}')

vol_dict = collections.defaultdict(list)
for path in test_npz:
    fname = os.path.basename(path)
    m = re.match(r'(patient\d+_frame\d+)_slice(\d+)\.npz', fname)
    if m:
        vol_dict[m.group(1)].append((int(m.group(2)), path))
    else:
        # 3D volume file (test set stored as full volumes)
        vol_dict[fname.replace('.npz', '')].append((0, path))
print(f'Volumes: {len(vol_dict)}')

all_metrics = []   # [(dc_rv, hd_rv, dc_myo, hd_myo, dc_lv, hd_lv), ...]

with torch.no_grad():
    for vol_key, slice_list in tqdm(sorted(vol_dict.items()), desc='Volumes'):
        slice_list.sort(key=lambda x: x[0])

        d0  = np.load(slice_list[0][1])
        key = 'img' if 'img' in d0 else 'image'

        if len(slice_list) == 1 and d0[key].ndim == 3:
            # Full 3D volume in one file
            img_vol = d0[key]
            lbl_vol = d0['label']
        else:
            imgs = [np.load(p)[key] for _, p in slice_list]
            lbls = [np.load(p)['label'] for _, p in slice_list]
            img_vol = np.stack(imgs)
            lbl_vol = np.stack(lbls)

        pred_vol = np.zeros_like(lbl_vol)
        for si in range(img_vol.shape[0]):
            slc  = img_vol[si]
            h, w = slc.shape
            if h != IMG_SIZE or w != IMG_SIZE:
                slc = zoom(slc, (IMG_SIZE / h, IMG_SIZE / w), order=3)
            t = torch.from_numpy(slc).unsqueeze(0).unsqueeze(0).float().cuda()
            with torch.cuda.amp.autocast():
                P = net_inf(t)
            out  = sum(P)
            out  = torch.softmax(out, dim=1).argmax(dim=1).squeeze(0).cpu().numpy()
            if h != IMG_SIZE or w != IMG_SIZE:
                out = zoom(out, (h / IMG_SIZE, w / IMG_SIZE), order=0)
            pred_vol[si] = out

        vol_m = []
        for c in range(1, NUM_CLASSES):
            p_c = pred_vol == c
            g_c = lbl_vol  == c
            vol_m += [_dc_vol(p_c, g_c), _hd95_vol(p_c, g_c)]
        all_metrics.append(vol_m)

M = np.array(all_metrics)   # (N_vols, 6)  [dc_rv, hd_rv, dc_myo, hd_myo, dc_lv, hd_lv]
print(f'\n=== Volume-Level Results  ({len(all_metrics)} volumes) ===')
for ci, cname in enumerate(CLASS_NAMES):
    dc_mean = M[:, ci * 2    ].mean() * 100
    hd_mean = M[:, ci * 2 + 1].mean()
    print(f'  {cname:<4}  Dice = {dc_mean:.2f}%   HD95 = {hd_mean:.2f} mm')
mean_dice = np.mean([M[:, ci * 2].mean() for ci in range(NUM_CLASSES - 1)]) * 100
print(f'  ----------------------------------------')
print(f'  Mean Dice  : {mean_dice:.2f}%  (paper target: 92.56%)')

In [ ]:
# Cell 17 — Test results: violin plot + bar chart (per-class Dice)
import matplotlib.pyplot as plt
import numpy as np

dice_per_class = [M[:, ci * 2] * 100 for ci in range(NUM_CLASSES - 1)]
colors_cls     = ['#e63946', '#f4a261', '#2a9d8f']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Violin plot with individual points ---
parts = axes[0].violinplot(
    dice_per_class,
    positions=range(1, NUM_CLASSES),
    widths=0.55,
    showmeans=False, showmedians=True, showextrema=False,
)
for body, col in zip(parts['bodies'], colors_cls):
    body.set_facecolor(col)
    body.set_alpha(0.55)
    body.set_edgecolor('black')
    body.set_linewidth(0.8)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(2)

# Scatter individual volumes on top
for ci, vals in enumerate(dice_per_class):
    jitter = np.random.uniform(-0.12, 0.12, size=len(vals))
    axes[0].scatter(np.full(len(vals), ci + 1) + jitter, vals,
                    color=colors_cls[ci], edgecolors='white',
                    linewidths=0.4, s=28, alpha=0.8, zorder=3)

axes[0].set_xticks(range(1, NUM_CLASSES))
axes[0].set_xticklabels(CLASS_NAMES, fontsize=11)
axes[0].set_title('Per-Class Dice Distribution (test volumes)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Dice (%)', fontsize=11)
axes[0].set_ylim(0, 105)
axes[0].grid(True, axis='y', alpha=0.3)
axes[0].axhline(mean_dice, color='black', linestyle='--', alpha=0.5,
                label=f'Mean {mean_dice:.2f}%')
axes[0].legend(fontsize=10)

# --- Bar chart with std ---
means = [np.mean(v) for v in dice_per_class]
stds  = [np.std(v)  for v in dice_per_class]
bars  = axes[1].bar(CLASS_NAMES, means, yerr=stds, capsize=5,
                    color=colors_cls, alpha=0.8,
                    error_kw=dict(ecolor='black', linewidth=1.5))
for bar, mean in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, mean + 1.5,
                 f'{mean:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

axes[1].set_title('Per-Class Dice: Mean ± Std (test volumes)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Dice (%)', fontsize=11)
axes[1].set_ylim(0, 110)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'test_dice_distribution.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')